In [9]:
from rdkit import Chem
import numpy as np
import csv
import joblib
import sys
from anytree import Node, RenderTree
sys.path.append('molecule-web-app/backend/rules')  # replace with your local path
sys.path.append('molecule-web-app/backend')
# Now you can import your module as usual
from rules import execute_step
import helper
from ml_helper import make_fingerprint

In [24]:
sm_smiles = 'CC(=O)Cl.O'
end_smiles = 'CC(=O)O.Cl'
sm = Chem.MolFromSmiles(sm_smiles)
end = Chem.MolFromSmiles(end_smiles)
depth_limit = 4
sm_node = Node("SM", mol=sm)

In [25]:
# Load the model
clf = joblib.load('models/v11/xg_model.pkl')

In [36]:
step_no = 0
nodes = [sm_node]
signal_break = False
while step_no < depth_limit:
    for int in nodes:
        nodes_to_next = []
        print(f'Layer {step_no+1}')
        product_mols, mechs, mol_inp  = execute_step("Placeholder", True, True, int.mol, False)
        for i, product in enumerate(product_mols):
            fp = np.array(make_fingerprint(mol_inp, mechs[i]))
            # Predict using the model
            pred = clf.predict(fp.reshape(1,-1))[0]
            if Chem.MolToInchi(product, options='Fixed-H') == Chem.MolToInchi(end, options='Fixed-H'):
                print('Found the target molecule!')
                print(f'The final step (mol {i}) was predicted: {pred}')
                new_node = Node(f"Mol {i}", parent=int, mol=product, mech=mechs[i])
                signal_break = True
                break
            elif pred == 1:
                new_node = Node(f"Mol {i}", parent=int, mol=product, mech=mechs[i])
                nodes_to_next.append(new_node)
        if signal_break:
            break
    if signal_break:
        break
    nodes = nodes_to_next
    step_no += 1

       


Layer 1
Found the target molecule!
The final step (mol 11) was predicted: 1


[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed

[15:58:28] WARNING: Proton(s) added/removed; Omitted undefined stereo

[15:58:28] WARNING: Proton(s) added/removed



In [37]:
len(make_fingerprint(mol_inp, mechs[1]))

160

In [38]:
for pre, fill, node in RenderTree(sm_node):
    print(f"{pre}{node.name} - {Chem.MolToSmiles(node.mol)}")
    if hasattr(node, 'mech'):
        print(f"{pre}Mechanism: {node.mech}")
    else:
        print(f"{pre}Mechanism: None")

SM - CC(=O)Cl.O
Mechanism: None
├── Mol 0 - C=C(O)Cl.O
├── Mechanism: [8, 2, 0, 1, 2]
├── Mol 4 - C=C=O.Cl.O
├── Mechanism: [8, 3, 0, 1, 3]
├── Mol 6 - C=C([O-])Cl.[OH3+]
├── Mechanism: [8, 4, 0, 1, 2]
├── Mol 8 - CC#[O+].O.[Cl-]
├── Mechanism: [1, 2, 1, None, 3]
├── Mol 10 - CC([O-])([OH2+])Cl
├── Mechanism: [1, 4, 1, None, 2]
├── Mol 0 - C=C(O)Cl.O
├── Mechanism: [8, 2, 0, 1, 2]
├── Mol 4 - C=C=O.Cl.O
├── Mechanism: [8, 3, 0, 1, 3]
├── Mol 6 - C=C([O-])Cl.[OH3+]
├── Mechanism: [8, 4, 0, 1, 2]
├── Mol 8 - CC#[O+].O.[Cl-]
├── Mechanism: [1, 2, 1, None, 3]
├── Mol 10 - CC([O-])([OH2+])Cl
├── Mechanism: [1, 4, 1, None, 2]
├── Mol 11 - CC(=O)[OH2+].[Cl-]
├── Mechanism: [1, 4, 1, None, 3]
├── Mol 0 - C=C(O)Cl.O
├── Mechanism: [8, 2, 0, 1, 2]
├── Mol 4 - C=C=O.Cl.O
├── Mechanism: [8, 3, 0, 1, 3]
├── Mol 6 - C=C([O-])Cl.[OH3+]
├── Mechanism: [8, 4, 0, 1, 2]
├── Mol 8 - CC#[O+].O.[Cl-]
├── Mechanism: [1, 2, 1, None, 3]
├── Mol 10 - CC([O-])([OH2+])Cl
├── Mechanism: [1, 4, 1, None, 2]
├── Mol 

In [29]:
sm_node.children

(Node('/SM/Mol 0', mech=[8, 2, 0, 1, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c4ebf40>),
 Node('/SM/Mol 4', mech=[8, 3, 0, 1, 3], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c368040>),
 Node('/SM/Mol 6', mech=[8, 4, 0, 1, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c3680b0>),
 Node('/SM/Mol 8', mech=[1, 2, 1, None, 3], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c3682e0>),
 Node('/SM/Mol 10', mech=[1, 4, 1, None, 2], mol=<rdkit.Chem.rdchem.Mol object at 0x7f934c368200>))